In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLE_DIR    = PROJECT_ROOT / "outputs" / "comparison_tables"

# All comparison graphs saved here
COMP_DIR = PROJECT_ROOT / "outputs" / "comparison_graphs"
COMP_DIR.mkdir(parents=True, exist_ok=True)

print("Comparison graphs will be saved to:", COMP_DIR)

Comparison graphs will be saved to: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs


In [ ]:
from pathlib import Path
import pandas as pd

# 1. Explicitly set paths directly to the main outputs folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT = PROJECT_ROOT / "outputs"

# Helper function to find a file even if its casing varies on disk
def load_csv_safely(directory, filename_options):
    for opt in filename_options:
        file_path = directory / opt
        if file_path.exists():
            return pd.read_csv(file_path)
    # Direct try fallback if both fail to throw descriptive error
    return pd.read_csv(directory / filename_options[0])

# ── Baseline ───────────────────────────────────────────────────────────────
df_base_fe = load_csv_safely(TABLE_DIR, ["baseline_results_with_fe.csv", "with_FE_Baseline_results.csv"])
df_base_no = load_csv_safely(TABLE_DIR, ["baseline_results_without_fe.csv", "without_FE_Baseline_results.csv"])

# ── Tuned ──────────────────────────────────────────────────────────────────
df_tune_fe = load_csv_safely(TABLE_DIR, ["tuned_results_with_fe.csv", "with_FE_tuned_results.csv"])
df_tune_no = load_csv_safely(TABLE_DIR, ["tuned_results_without_fe.csv", "without_FE_tuned_results.csv"])

# ── Standardise column names so all 4 match ────────────────────────────────
def standardise_tuned(df, label):
    df = df.copy()
    rename = {
        "f1"       : "Test F1",
        "roc_auc"  : "Test ROC-AUC",
        "accuracy" : "Test Accuracy",
        "precision": "Test Precision",
        "recall"   : "Test Recall",
    }
    df = df.rename(columns=rename)
    df["Experiment"] = label
    return df

def standardise_baseline(df, label):
    df = df.copy()
    df["Experiment"] = label
    if "tune_time_sec"  not in df.columns: df["tune_time_sec"]  = 0.0
    if "total_time_sec" not in df.columns: df["total_time_sec"] = df.get("train_time_sec", 0.0)
    return df

df_base_fe = standardise_baseline(df_base_fe, "with_FE_baseline")
df_base_no = standardise_baseline(df_base_no, "without_FE_baseline")
df_tune_fe = standardise_tuned(df_tune_fe,    "with_FE_tuned")
df_tune_no = standardise_tuned(df_tune_no,    "without_FE_tuned")

# ── Model list intersection ───────────────────────────────────────────────
common_models = sorted(list(
    set(df_base_fe["Model"]) & 
    set(df_base_no["Model"]) & 
    set(df_tune_fe["Model"]) & 
    set(df_tune_no["Model"])
))

print(f"Common models across all 4 experiments: {len(common_models)}")
print(common_models)


Common models across all 4 experiments: 14
['AdaBoost', 'Bagging', 'Decision Tree', 'Gaussian NB', 'Gradient Boosting', 'KNN', 'LDA', 'Logistic Regression', 'Perceptron', 'QDA', 'Random Forest', 'SVM', 'Stacking ML', 'XGBoost']


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Point dynamically to the comparison_graphs subfolder seen in your sidebar
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT = PROJECT_ROOT / "outputs"
GRAPH_DIR = BASE_OUT / "comparison_graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

# Shared plotting variables
models_plot = [m for m in common_models if m != "Stacking ML"]
x = np.arange(len(models_plot))

def save_fig(fig_obj, filename):
    out_path = GRAPH_DIR / filename
    fig_obj.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig_obj)
    print(f"✓ Saved → {out_path}")


In [5]:
fe_times = df_base_fe.set_index("Model").loc[models_plot, "train_time_sec"].values
no_times = df_base_no.set_index("Model").loc[models_plot, "train_time_sec"].values

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.plot(x, no_times, marker="o", linewidth=2, label="without FE (41 features)", color="#d62728")
ax.plot(x, fe_times, marker="s", linewidth=2, label="with FE (11 features)", color="#1f77b4")

for idx, (nt, ft) in enumerate(zip(no_times, fe_times)):
    ax.annotate(f"{nt:.3f}s", (idx, nt), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=7.5, color="#d62728")
    ax.annotate(f"{ft:.3f}s", (idx, ft), textcoords="offset points", xytext=(0, -13), ha="center", fontsize=7.5, color="#1f77b4")

ax.set_xticks(x)
ax.set_xticklabels(models_plot, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Training Time (seconds)", fontsize=10)
ax.set_title("Baseline Models — Training Time\nwith FE vs without FE", fontsize=12, fontweight="bold")
ax.legend(fontsize=10, loc="upper right")
plt.tight_layout()
save_fig(fig, "graph1_baseline_training_time_fe_vs_nofe.png")


✓ Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs\graph1_baseline_training_time_fe_vs_nofe.png


In [6]:
fe_times_t = df_tune_fe.set_index("Model").loc[models_plot, "train_time_sec"].values
no_times_t = df_tune_no.set_index("Model").loc[models_plot, "train_time_sec"].values

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.plot(x, no_times_t, marker="o", linewidth=2, label="without FE (41 features)", color="#d62728")
ax.plot(x, fe_times_t, marker="s", linewidth=2, label="with FE (11 features)", color="#1f77b4")

for idx, (nt, ft) in enumerate(zip(no_times_t, fe_times_t)):
    ax.annotate(f"{nt:.3f}s", (idx, nt), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=7.5, color="#d62728")
    ax.annotate(f"{ft:.3f}s", (idx, ft), textcoords="offset points", xytext=(0, -13), ha="center", fontsize=7.5, color="#1f77b4")

ax.set_xticks(x)
ax.set_xticklabels(models_plot, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Training Time (seconds)", fontsize=10)
ax.set_title("Tuned Models — Training Time\nwith FE vs without FE", fontsize=12, fontweight="bold")
ax.legend(fontsize=10, loc="upper right")
plt.tight_layout()
save_fig(fig, "graph2_tuned_training_time_fe_vs_nofe.png")


✓ Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs\graph2_tuned_training_time_fe_vs_nofe.png


In [7]:
fe_tune = df_tune_fe.set_index("Model").loc[models_plot, "tune_time_sec"].values
no_tune = df_tune_no.set_index("Model").loc[models_plot, "tune_time_sec"].values

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.plot(x, no_tune, marker="o", linewidth=2, label="without FE (41 features)", color="#d62728")
ax.plot(x, fe_tune, marker="s", linewidth=2, label="with FE (11 features)", color="#1f77b4")

for idx, (nt, ft) in enumerate(zip(no_tune, fe_tune)):
    ax.annotate(f"{nt:.1f}s", (idx, nt), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=7.5, color="#d62728")
    ax.annotate(f"{ft:.1f}s", (idx, ft), textcoords="offset points", xytext=(0, -13), ha="center", fontsize=7.5, color="#1f77b4")

ax.set_xticks(x)
ax.set_xticklabels(models_plot, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Optuna Tuning Time (seconds)", fontsize=10)
ax.set_title("Tuned Models — Optuna Hyperparameter Search Time\nwith FE vs without FE", fontsize=12, fontweight="bold")
ax.legend(fontsize=10, loc="upper right")
plt.tight_layout()
save_fig(fig, "graph3_tuning_time_fe_vs_nofe.png")


✓ Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs\graph3_tuning_time_fe_vs_nofe.png


In [9]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Explicit Path & Variables Initialization (Fixes the NameError)
dfs = [df_base_no, df_base_fe, df_tune_no, df_tune_fe]
labels = ["without_FE_baseline", "with_FE_baseline", "without_FE_tuned", "with_FE_tuned"]
colors_line = ["#aec7e8", "#1f77b4", "#ffbb78", "#ff7f0e"]
markers_line = ["o", "v", "^", "s"]
stagger_offsets = [-18, -6, 6, 18]

# Setup model plotting indices
models_plot = [m for m in common_models if m != "Stacking ML"]
x = np.arange(len(models_plot))

# 2. Plotting Execution Loop
fig, ax = plt.subplots(figsize=(16, 7))

for i, (df, label, color, marker) in enumerate(zip(dfs, labels, colors_line, markers_line)):
    vals = df.set_index("Model").loc[models_plot, "Test F1"].values
    ax.plot(x, vals, marker=marker, linewidth=2, label=label, color=color)
    
    y_shift = stagger_offsets[i]
    for idx, v in enumerate(vals):
        ax.annotate(
            f"{v:.3f}", 
            (idx, v), 
            textcoords="offset points", 
            xytext=(0, y_shift), 
            ha="center", 
            fontsize=7,
            color="#2c3e50",  # Professional dark charcoal text
            fontweight="bold" if "tuned" in label else "normal",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75)
        )

ax.set_xticks(x)
ax.set_xticklabels(models_plot, rotation=30, ha="right", fontsize=9.5)
ax.set_ylabel("Test F1 Score", fontsize=10.5)
ax.set_ylim(0.38, 1.06)
ax.set_title("F1 Score Trends Across All 4 Experiments per Model", fontsize=12, fontweight="bold")
ax.legend(fontsize=9.5, loc="lower right")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

# Save using your predefined utility
save_fig(fig, "graph4_f1_all_experiments.png")


✓ Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs\graph4_f1_all_experiments.png


In [10]:
fig, ax = plt.subplots(figsize=(16, 7))

# Distinct vertical offsets to separate the 4 lines cleanly
stagger_offsets = [-18, -6, 6, 18]

for i, (df, label, color, marker) in enumerate(zip(dfs, labels, colors_line, markers_line)):
    vals = df.set_index("Model").loc[models_plot, "Test ROC-AUC"].values
    ax.plot(x, vals, marker=marker, linewidth=2, label=label, color=color)
    
    y_shift = stagger_offsets[i]
    for idx, v in enumerate(vals):
        ax.annotate(
            f"{v:.3f}", 
            (idx, v), 
            textcoords="offset points", 
            xytext=(0, y_shift), 
            ha="center", 
            fontsize=7,
            color="#2c3e50",  # Professional dark charcoal text
            fontweight="bold" if "tuned" in label else "normal",
            # White soft capsule background prevents text-on-line clashing
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75)
        )

ax.set_xticks(x)
ax.set_xticklabels(models_plot, rotation=30, ha="right", fontsize=9.5)
ax.set_ylabel("Test ROC-AUC Score", fontsize=10.5)
ax.set_ylim(0.48, 1.06)
ax.set_title("ROC-AUC Trends Across All 4 Experiments per Model", fontsize=12, fontweight="bold")
ax.legend(fontsize=9.5, loc="lower right")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
save_fig(fig, "graph5_rocauc_all_experiments.png")


✓ Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs\graph5_rocauc_all_experiments.png


In [11]:
fig, ax = plt.subplots(figsize=(10, 6))

for df, label, color, marker in zip(dfs, labels, colors_line, markers_line):
    temp = df.set_index("Model").loc[models_plot]
    ax.scatter(temp["total_time_sec"], temp["Test F1"], s=80, color=color, marker=marker, label=label, alpha=0.85)

ax.set_xlabel("Total Computation Time (log scale, seconds)", fontsize=10)
ax.set_ylabel("Test F1 Score", fontsize=10)
ax.set_xscale("log")  # Keeps hyperparameter vs base time comparable
ax.set_title("Model Efficiency: Computation Time vs Test F1 Score", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, loc="lower left")
plt.tight_layout()
save_fig(fig, "graph6_time_vs_f1_scatter.png")


✓ Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs\graph6_time_vs_f1_scatter.png


In [12]:
summary_rows = []
for df, label in zip(dfs, labels):
    summary_rows.append({
        "Experiment": label,
        "Mean Accuracy": df["Test Accuracy"].mean(),
        "Mean Precision": df["Test Precision"].mean(),
        "Mean Recall": df["Test Recall"].mean(),
        "Mean F1": df["Test F1"].mean(),
        "Mean ROC-AUC": df["Test ROC-AUC"].mean()
    })

df_summary = pd.DataFrame(summary_rows).round(4)

# Create standard text table layout drawn directly into matplotlib canvas
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis("off")
table_obj = ax.table(cellText=df_summary.values, colLabels=df_summary.columns, cellLoc="center", loc="center")
table_obj.auto_set_font_size(False)
table_obj.set_fontsize(9)
table_obj.scale(1.1, 1.4)
ax.set_title("Summary Overview: Average Performance Scores Across Experiments", fontsize=11, fontweight="bold")
plt.tight_layout()
save_fig(fig, "graph7_average_metrics_summary_table.png")


✓ Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs\graph7_average_metrics_summary_table.png


In [13]:
avg_times = [df["train_time_sec"].mean() for df in dfs]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels, avg_times, color=["#aec7e8", "#1f77b4", "#ffbb78", "#ff7f0e"], width=0.5, edgecolor="gray", alpha=0.85)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + (max(avg_times) * 0.01),
            f"{h:.4f}s", ha="center", va="bottom", fontsize=9, fontweight="bold", color="dimgray")

ax.set_ylabel("Average Train Time (seconds)", fontsize=10)
ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_title("Overall Baseline vs Tuned Speed: Average Training Duration Comparison", fontsize=12, fontweight="bold")
plt.tight_layout()
save_fig(fig, "graph8_average_training_time_summary.png")


✓ Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\comparison_graphs\graph8_average_training_time_summary.png


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Configuration & Paths (Dynamic assignment prevents absolute reference crashes)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT = PROJECT_ROOT / "outputs"
GRAPH_DIR = BASE_OUT / "comparison_graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)


# Define the 4 experiment types and their corresponding DataFrames
experiments_list = [
    {"df": df_base_no, "title": "Without FE - Baseline", "filename": "bar_metrics_without_FE_baseline.png"},
    {"df": df_base_fe, "title": "With FE - Baseline",    "filename": "bar_metrics_with_FE_baseline.png"},
    {"df": df_tune_no, "title": "Without FE - Tuned",    "filename": "bar_metrics_without_FE_tuned.png"},
    {"df": df_tune_fe, "title": "With FE - Tuned",       "filename": "bar_metrics_with_FE_tuned.png"}
]

# Metrics to extract for each model
metrics_to_plot = ["Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC"]
colors_bars = ["#4a90e2", "#2ecc71", "#e74c3c", "#9b59b6", "#f1c40f"] # Distinct professional colors

# Setup plot positioning variables
models_list = [m for m in common_models if m != "Stacking ML"]
x_coords = np.arange(len(models_list))
bar_width = 0.15  # Spreading 5 bars across each model index

# 2. Loop through each experiment type and draw its grouped bar chart
for exp in experiments_list:
    df_active = exp["df"].set_index("Model").loc[models_list]
    
    fig, ax = plt.subplots(figsize=(16, 7))
    
    # Plot each metric bar with a precise layout offset
    for idx, metric in enumerate(metrics_to_plot):
        metric_values = df_active[metric].values
        offset = (idx - len(metrics_to_plot) / 2 + 0.5) * bar_width
        ax.bar(x_coords + offset, metric_values, width=bar_width, label=metric, color=colors_bars[idx], edgecolor="none")
        
    # Formatting styles matching your reference layout
    ax.set_xticks(x_coords)
    ax.set_xticklabels(models_list, rotation=30, ha="right", fontsize=9.5)
    ax.set_ylabel("Percentage Value / Score", fontsize=11)
    ax.set_ylim(0.0, 1.15) # Room at the top for labels if needed
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y*100)}%' if y <= 1.0 else f'{y}'))
    
    ax.set_title(f"Models Performance Metrics Comparison\n({exp['title']})", fontsize=13, fontweight="bold")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=5, fontsize=10, frameon=True)
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    
    # Save directly to your target directory
    save_path = GRAPH_DIR / exp["filename"]
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Generated & Saved → {save_path.name}")

print("\nAll 4 grouped metric bar charts have been successfully stored in your comparison directory.")


✓ Generated & Saved → bar_metrics_without_FE_baseline.png
✓ Generated & Saved → bar_metrics_with_FE_baseline.png
✓ Generated & Saved → bar_metrics_without_FE_tuned.png
✓ Generated & Saved → bar_metrics_with_FE_tuned.png

All 4 grouped metric bar charts have been successfully stored in your comparison directory.


In [ ]:
from pathlib import Path

# 1. Setup the explicit output path pointing to your outputs directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT = PROJECT_ROOT / "outputs"
MODELS_DIR = PROJECT_ROOT / "models"

# 2. Create a dedicated folder for all your Explainable AI outputs
XAI_DIR = BASE_OUT / "explainable_ai"
XAI_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Dedicated Explainable AI workspace folder created at:\n  {XAI_DIR}")


✓ Dedicated Explainable AI workspace folder created at:
  c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\explainable_ai


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# ── 1. DEFINING PROJECT DIRECTORIES & WORKSPACES ───────────────────────────
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT     = PROJECT_ROOT / "outputs"
MODELS_DIR    = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Dedicated workspace folder for your explainable AI figures
XAI_DIR = BASE_OUT / "explainable_ai"
XAI_DIR.mkdir(parents=True, exist_ok=True)

# ── 2. AUTOMATIC MEMORY RESTORATION LAYER ──────────────────────────────────
try:
    if 'X_test_fe' not in locals() or 'X_train_fe' not in locals():
        print("🔄 Test variables missing from memory. Re-loading datasets safely from disk...")
        X_test_fe  = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
        X_train_fe = pd.read_csv(PROCESSED_DIR / "X_train_selected_imp.csv")
    print(f"📊 Dataset successfully anchored into workspace: {X_test_fe.shape[1]} features.")
except Exception as e:
    print(f"⚠️ Could not automatically load datasets from {PROCESSED_DIR}: {e}")

# ── 3. RUN SHAP VISUAL ANALYSIS ────────────────────────────────────────────
try:
    import shap
    
    # Locate the best Random Forest tuned file
    model_path = MODELS_DIR / "with_fe_tuned_random_forest.joblib"
    if not model_path.exists():
        model_path = MODELS_DIR / "with_fe_tuned_random_forest_tuned_model.joblib"
        
    if not model_path.exists():
        raise FileNotFoundError(f"Missing tuned model file in directory: {MODELS_DIR}")
        
    model = joblib.load(model_path)
    
    # Safely unwrap CalibratedClassifierCV wrapper if present
    if hasattr(model, "calibrated_classifiers_") and len(model.calibrated_classifiers_) > 0:
        base_estimator = model.calibrated_classifiers_[0].estimator
    elif hasattr(model, "estimator"):
        base_estimator = model.estimator
    else:
        base_estimator = model

    # FIXED: Initialize TreeExplainer and force check_additivity=False to handle tree float variations
    explainer = shap.TreeExplainer(base_estimator, X_train_fe)
    shap_values_obj = explainer(X_test_fe, check_additivity=False)
    
    # Isolate class 1 (PCOS Positive) indicators for binary formatting splits
    if len(shap_values_obj.shape) == 3 and shap_values_obj.shape[-1] == 2:
        active_shap_values = shap_values_obj[:, :, 1]
    else:
        active_shap_values = shap_values_obj

    # GRAPH 1: Global Summary Plot (Beeswarm)
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.plots.beeswarm(active_shap_values, show=False)
    plt.title("SHAP Global Summary: Feature Impact Patterns on PCOS Risk", fontsize=12, fontweight="bold", pad=15)
    plt.tight_layout()
    
    shap_global_path = XAI_DIR / "shap_global_summary.png"
    fig.savefig(shap_global_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Saved Graph 1 (SHAP Global Summary) → {shap_global_path.name}")

    # GRAPH 2: Local Patient Explanation (Waterfall)
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.plots.waterfall(active_shap_values[0], show=False)
    plt.title("SHAP Local Explanation: Feature Contributions for Patient #1", fontsize=12, fontweight="bold", pad=15)
    plt.tight_layout()
    
    shap_local_path = XAI_DIR / "shap_local_waterfall.png"
    fig.savefig(shap_local_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Saved Graph 2 (SHAP Local Waterfall) → {shap_local_path.name}")

except ImportError:
    print("⚠ 'shap' library missing. Run: !pip install shap")
except Exception as e:
    print(f"⚠ Could not generate SHAP plots: {e}")

# ── 4. RUN LIME SURROGATE ASSESSMENT ────────────────────────────────────────
try:
    from lime.lime_tabular import LimeTabularExplainer
    
    lime_explainer = LimeTabularExplainer(
        training_data=np.array(X_train_fe),
        feature_names=list(X_train_fe.columns),
        class_names=["No PCOS", "PCOS"],
        mode="classification",
        random_state=42
    )
    
    # FIXED: Convert data_row to a clean, flat 1D numpy array (.values) to clear index errors
    patient_instance = X_test_fe.iloc[0].values
    
    exp = lime_explainer.explain_instance(
        data_row=patient_instance,
        predict_fn=model.predict_proba,
        num_features=len(X_test_fe.columns)
    )
    
    # GRAPH 3: Local Feature Weight Bars
    fig = exp.as_pyplot_figure()
    plt.title("LIME Local Explanation: Feature Weight Thresholds for Patient #1", fontsize=11, fontweight="bold", pad=15)
    plt.tight_layout()
    
    lime_path = XAI_DIR / "lime_local_explanation.png"
    fig.savefig(lime_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Saved Graph 3 (LIME Local Explanation) → {lime_path.name}")
    
except ImportError:
    print("⚠ 'lime' library missing. Run: !pip install lime")
except Exception as e:
    print(f"⚠ Could not generate LIME plot: {e}")


Background dataset has 378 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=378 when initializing the masker.


📊 Dataset successfully anchored into workspace: 11 features.
✓ Saved Graph 1 (SHAP Global Summary) → shap_global_summary.png
✓ Saved Graph 2 (SHAP Local Waterfall) → shap_local_waterfall.png
✓ Saved Graph 3 (LIME Local Explanation) → lime_local_explanation.png


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import precision_recall_curve, auc, f1_score

# ── 1. DEFINE PROJECT DIRECTORIES & WORKSPACES ───────────────────────────
PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT      = PROJECT_ROOT / "outputs"
MODELS_DIR    = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


PR_DIR = BASE_OUT / "precision_recall"
PR_DIR.mkdir(parents=True, exist_ok=True)

# ── 2. DATA RE-LOADER LAYER ───────────────────────────────────────────────
if 'X_test_fe' not in locals() or 'y_test' not in locals():
    X_test_fe = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
    y_test    = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

# ── 3. DYNAMIC METRIC GENERATION FOR ALL MODELS ───────────────────────────
fig, ax = plt.subplots(figsize=(10, 8)) # Made slightly taller to fit a larger legend cleanly
stats_rows = []

# Automatically evaluate every model from your project list
# (If common_models isn't active, it falls back to a comprehensive list of your models)
if 'common_models' in locals():
    models_to_evaluate = common_models
else:
    models_to_evaluate = [
        "Logistic Regression", "Decision Tree", "Random Forest", "SVM",
        "Gaussian NB", "Bagging", "AdaBoost", "Gradient Boosting",
        "KNN", "LDA", "QDA", "Perceptron", "XGBoost"
    ]

# Use a professional color map for multiple lines
colormap = plt.cm.get_cmap("tab20", len(models_to_evaluate))

for idx, name in enumerate(models_to_evaluate):
    safe = name.lower().replace(" ", "_").replace("/", "_")
    model_path = MODELS_DIR / f"with_fe_tuned_{safe}.joblib"
    if not model_path.exists():
        model_path = MODELS_DIR / f"with_fe_tuned_{safe}_tuned_model.joblib"
        
    # Safely skip models that weren't tuned or saved to disk
    if not model_path.exists():
        continue
        
    model = joblib.load(model_path)
    proba = model.predict_proba(X_test_fe)[:, 1]
    
    # 1. Plot individual PR Curve trend line
    precision, recall, _ = precision_recall_curve(y_test, proba)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, linewidth=1.5, label=f"{name} (AUC = {pr_auc:.4f})", color=colormap(idx))
    
    # 2. Compute 95% Confidence Interval for F1-score via Bootstrapping
    rng = np.random.default_rng(42)
    bootstrapped_f1 = []
    y_test_arr = np.array(y_test)
    preds = (proba >= 0.5).astype(int)
    
    for _ in range(500):
        indices = rng.choice(len(y_test), size=len(y_test), replace=True)
        if len(np.unique(y_test_arr[indices])) < 2:
            continue
        bootstrapped_f1.append(f1_score(y_test_arr[indices], preds[indices], zero_division=0))
        
    low_ci = np.percentile(bootstrapped_f1, 2.5)
    high_ci = np.percentile(bootstrapped_f1, 97.5)
    stats_rows.append(f"{name} F1 95% CI: [{low_ci:.4f}, {high_ci:.4f}]\n")

# Formatting layouts for many lines
ax.set_xlabel("Recall (Sensitivity)", fontsize=10)
ax.set_ylabel("Precision (Positive Predictive Value)", fontsize=10)
ax.set_title("Precision-Recall Curves (Comprehensive Model Evaluation)", fontsize=12, fontweight="bold")

# Place the larger legend cleanly to the right side so it doesn't overlap the lines
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8.5, frameon=True)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()

# Save updated files
fig_path = PR_DIR / "precision_recall_curves.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight") # Added bbox_inches to prevent legend clipping
plt.close(fig)

text_path = PR_DIR / "f1_confidence_intervals.txt"
text_path.write_text("".join(stats_rows))

print(f"✓ Re-generated with all available models! Saved to: {PR_DIR.name}")


✓ Re-generated with all available models! Saved to: precision_recall


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# ── 1. DEFINE PROJECT DIRECTORIES & WORKSPACES ───────────────────────────
PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT      = PROJECT_ROOT / "outputs"
MODELS_DIR    = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


# Explicitly define and build your CAL_DIR (Fixes the NameError)
CAL_DIR = BASE_OUT / "calibration"
CAL_DIR.mkdir(parents=True, exist_ok=True)

# ── 2. DATA RE-LOADER LAYER ───────────────────────────────────────────────
if 'X_test_fe' not in locals() or 'y_test' not in locals():
    X_test_fe = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
    y_test    = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

# ── 3. DYNAMIC CALIBRATION FOR ALL AVAILABLE MODELS ───────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
brier_logs = []

# Fetch all models automatically
if 'common_models' in locals():
    models_to_evaluate = common_models
else:
    models_to_evaluate = [
        "Logistic Regression", "Decision Tree", "Random Forest", "SVM",
        "Gaussian NB", "Bagging", "AdaBoost", "Gradient Boosting",
        "KNN", "LDA", "QDA", "Perceptron", "XGBoost"
    ]

# Setup a clean color spectrum map
colormap = plt.cm.get_cmap("tab20", len(models_to_evaluate))

for idx, name in enumerate(models_to_evaluate):
    safe = name.lower().replace(" ", "_").replace("/", "_")
    model_path = MODELS_DIR / f"with_fe_tuned_{safe}.joblib"
    if not model_path.exists():
        model_path = MODELS_DIR / f"with_fe_tuned_{safe}_tuned_model.joblib"
        
    # Safely skip ungenerated files
    if not model_path.exists():
        continue
        
    model = joblib.load(model_path)
    proba = model.predict_proba(X_test_fe)[:, 1]
    
    # Calculate calibration bins and Brier scores
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=5)
    brier = brier_score_loss(y_test, proba)
    
    ax.plot(prob_pred, prob_true, marker="s", markersize=4, linewidth=1.5, 
            label=f"{name} (Brier = {brier:.4f})", color=colormap(idx))
    brier_logs.append(f"{name} Brier Score: {brier:.5f}\n")

# ── 4. VISUAL FORMATTING & EXPORT ──────────────────────────────────────────
ax.set_xlabel("Mean Predicted Probability", fontsize=10)
ax.set_ylabel("Fraction of Positives (Actual Frequency)", fontsize=10)
ax.set_title("Probability Calibration Curves (Risk Score Reliability)", fontsize=12, fontweight="bold")

# Align legend cleanly to the right side
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8.5, frameon=True)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()

# Save the final publication-ready structures
fig_path = CAL_DIR / "probability_calibration_curves.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.close(fig)

text_path = CAL_DIR / "brier_scores_summary.txt"
text_path.write_text("".join(brier_logs))

print(f"✓ Calibration chart successfully saved → {fig_path.name}")
print(f"✓ Brier scores summary text sheet stored → {text_path.name}")
print(f"📁 Folder path: {CAL_DIR}")


✓ Calibration chart successfully saved → probability_calibration_curves.png
✓ Brier scores summary text sheet stored → brier_scores_summary.txt
📁 Folder path: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\calibration


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# ── 1. DEFINE PROJECT DIRECTORIES & WORKSPACES ───────────────────────────
PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT      = PROJECT_ROOT / "outputs"
MODELS_DIR    = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Explicitly define and build your DCA_DIR (Fixes the NameError)
DCA_DIR = BASE_OUT / "decision_curve"
DCA_DIR.mkdir(parents=True, exist_ok=True)

# ── 2. DATA RE-LOADER LAYER ───────────────────────────────────────────────
if 'X_test_fe' not in locals() or 'y_test' not in locals():
    X_test_fe = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
    y_test    = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

# ── 3. CLINICAL DECISION CURVE EVALUATION (DCA) ───────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
thresholds = np.linspace(0.01, 0.99, 100)

n_patients = len(y_test)
pt_actual_pos = np.sum(y_test)

# Baseline strategies
net_benefit_all = (pt_actual_pos / n_patients) - (1 - (pt_actual_pos / n_patients)) * (thresholds / (1 - thresholds))
ax.plot(thresholds, net_benefit_all, "g--", linewidth=1.5, label="Treat All Patients", alpha=0.6)
ax.plot(thresholds, np.zeros_like(thresholds), "k-", linewidth=1.5, label="Treat None (No Screening)", alpha=0.7)

# Fetch all available models automatically
if 'common_models' in locals():
    models_to_evaluate = common_models
else:
    models_to_evaluate = [
        "Logistic Regression", "Decision Tree", "Random Forest", "SVM",
        "Gaussian NB", "Bagging", "AdaBoost", "Gradient Boosting",
        "KNN", "LDA", "QDA", "Perceptron", "XGBoost"
    ]

# Setup color scheme spectrum map
colormap = plt.cm.get_cmap("tab20", len(models_to_evaluate))

for idx, name in enumerate(models_to_evaluate):
    safe = name.lower().replace(" ", "_").replace("/", "_")
    model_path = MODELS_DIR / f"with_fe_tuned_{safe}.joblib"
    if not model_path.exists():
        model_path = MODELS_DIR / f"with_fe_tuned_{safe}_tuned_model.joblib"
        
    # Safely skip ungenerated model files
    if not model_path.exists():
        continue
        
    model = joblib.load(model_path)
    proba = model.predict_proba(X_test_fe)[:, 1]
    
    net_benefits = []
    for t in thresholds:
        preds = (proba >= t).astype(int)
        tp = np.sum((preds == 1) & (y_test == 1))
        fp = np.sum((preds == 1) & (y_test == 0))
        
        # Standard Net Benefit clinical formula calculation
        nb = (tp / n_patients) - (fp / n_patients) * (t / (1 - t))
        net_benefits.append(nb)
        
    ax.plot(thresholds, net_benefits, linewidth=1.5, label=f"Using {name} Model", color=colormap(idx))

# ── 4. VISUAL FORMATTING & EXPORT ──────────────────────────────────────────
ax.set_xlim(0, 0.8)
ax.set_ylim(-0.05, max(net_benefit_all) + 0.1)
ax.set_xlabel("Probability Threshold Level ($p_t$)", fontsize=10)
ax.set_ylabel("Clinical Net Benefit", fontsize=10)
ax.set_title("Decision Curve Analysis (DCA): Clinical Net Diagnostic Utility", fontsize=12, fontweight="bold")

# Move the comprehensive legend box cleanly out to the right side
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8.5, frameon=True)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()

# Save the final publication-ready graph
fig_path = DCA_DIR / "decision_curve_analysis.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"✓ Clinical Decision Curve Analysis Plot successfully saved → {fig_path.name}")
print(f"📁 Folder path: {DCA_DIR}")


✓ Clinical Decision Curve Analysis Plot successfully saved → decision_curve_analysis.png
📁 Folder path: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\decision_curve


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import confusion_matrix, cohen_kappa_score

# ── 1. DEFINE PROJECT DIRECTORIES & WORKSPACES ───────────────────────────
PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT      = PROJECT_ROOT / "outputs"
MODELS_DIR    = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


# ── 2. DATA & MODEL RE-LOADER LAYER (Fixes NameErrors) ─────────────────────
if 'y_test' not in locals() or 'X_test_fe' not in locals():
    print("🔄 Variables missing from memory. Re-loading datasets safely from disk...")
    X_test_fe = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
    y_test    = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

# Re-load your top calibrated model and find its threshold to get final_preds
model_path = MODELS_DIR / "with_fe_tuned_random_forest.joblib"
if not model_path.exists():
    model_path = MODELS_DIR / "with_fe_tuned_random_forest_tuned_model.joblib"

if model_path.exists():
    model = joblib.load(model_path)
    test_proba = model.predict_proba(X_test_fe)[:, 1]
    
    # Check if best_t exists in memory, otherwise default to a balanced threshold
    t_thresh = best_t if 'best_t' in locals() else 0.50
    final_preds = (test_proba >= t_thresh).astype(int)
else:
    print("⚠️ Tuned model file not found. Ensure with_fe_tuned_random_forest.joblib exists.")

# ── 3. EXECUTE ADVANCED  METRICS ────────────────────────────────────
print("\n" + "="*50)
print("=== ADVANCED JOURNAL MANUSCRIPT METRICS ===")
print("="*50 + "\n")

try:
    # Medical Diagnostic Metrics (Sensitivity & Specificity)
    tn, fp, fn, tp = confusion_matrix(y_test, final_preds).ravel()
    sensitivity = tp / (tp + fn)  # True Positive Rate (Recall)
    specificity = tn / (tn + fp)  # True Negative Rate

    print(f"🩺 Clinical Sensitivity (Recall) : {sensitivity:.4f}")
    print(f"🩺 Clinical Specificity         : {specificity:.4f}")

    # Balanced Accuracy (Crucial for medical class imbalance)
    balanced_acc = (sensitivity + specificity) / 2
    print(f" Balanced Accuracy            : {balanced_acc:.4f}")

    # Cohen's Kappa Coefficient (Proves model agreement isn't just luck)
    kappa = cohen_kappa_score(y_test, final_preds)
    print(f"🤝 Cohen's Kappa Score          : {kappa:.4f}")

    # Diagnostic Odds Ratio (DOR)
    dor = ((tp + 0.5) * (tn + 0.5)) / ((fp + 0.5) * (fn + 0.5))
    print(f"📈 Diagnostic Odds Ratio (DOR)  : {dor:.4f}")
    
    # Save these metrics directly to a clean text report in your outputs
    report_path = BASE_OUT / "advanced_metrics.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("=== ADVANCED METRICS ===\n\n")
        f.write(f"Clinical Sensitivity (Recall) : {sensitivity:.4f}\n")
        f.write(f"Clinical Specificity         : {specificity:.4f}\n")
        f.write(f"Balanced Accuracy            : {balanced_acc:.4f}\n")
        f.write(f"Cohen's Kappa Score          : {kappa:.4f}\n")
        f.write(f"Diagnostic Odds Ratio (DOR)  : {dor:.4f}\n")
    print(f"\n📑 Verification report successfully saved to disk → outputs/{report_path.name}")

except Exception as e:
    print(f"⚠️ Error computing advanced metrics: {e}")



=== ADVANCED JOURNAL MANUSCRIPT METRICS ===

🩺 Clinical Sensitivity (Recall) : 0.7925
🩺 Clinical Specificity         : 0.9364
 Balanced Accuracy            : 0.8644
🤝 Cohen's Kappa Score          : 0.7434
📈 Diagnostic Odds Ratio (DOR)  : 51.0000

📑 Verification report successfully saved to disk → outputs/advanced_metrics.txt
